# 01 — FAISS baseline từ CLIPFeatures BTC phát sẵn

**Phụ trách:** Tuấn Anh (N1) · **Mốc:** M1 · **GPU:** KHÔNG cần (CPU đủ)

Đầu vào: thư mục CLIPFeatures của BTC trên Drive chung.
Đầu ra: `faiss_clip.index` + `keyframe_ids.npy` lưu về Drive.

> Mở notebook này từ GitHub: Colab → File → Open notebook → GitHub → dán URL repo.

In [1]:
# === SETUP: mount Drive + cài thư viện ===
from google.colab import drive
drive.mount('/content/drive')
%pip -q install faiss-cpu numpy pandas

# === CONFIG: sửa đường dẫn theo Drive chung của đội ===
DRIVE_ROOT = '/content/drive/MyDrive/AIC2026'   # thư mục chia sẻ chung
CLIP_DIR   = f'{DRIVE_ROOT}/data/clip_features'  # nơi để CLIPFeatures BTC phát
OUT_DIR    = f'{DRIVE_ROOT}/outputs/index_v1_clip'
import os; os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 16.1 MB/s eta 0:00:00


In [ ]:
import os, glob
BASE = '/content/drive/MyDrive/AIC2026/data'
OUT  = '/content/drive/MyDrive/AIC2026/outputs'

def tim(ten):                      # tự gỡ trường hợp thư mục lồng 1 cấp
    d = os.path.join(BASE, ten)
    if glob.glob(f'{d}/*.npy') or glob.glob(f'{d}/*.csv'): return d
    for sub in sorted(glob.glob(f'{d}/*')):
        if os.path.isdir(sub) and (glob.glob(f'{sub}/*.npy') or glob.glob(f'{sub}/*.csv')):
            return sub
    return d

CLIP_DIR, MAP_DIR = tim('clip_features'), tim('map-keyframes')
os.makedirs(OUT, exist_ok=True)
print('CLIP_DIR =', CLIP_DIR, f'({len(glob.glob(f"{CLIP_DIR}/*.npy"))} file .npy)')
print('MAP_DIR  =', MAP_DIR,  f'({len(glob.glob(f"{MAP_DIR}/*.csv"))} file .csv)')

In [2]:
import os, glob, numpy as np, pandas as pd

# --- 1. Đối chiếu hai thư mục ---
vid_npy = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{CLIP_DIR}/*.npy')}
vid_csv = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{MAP_DIR}/*.csv')}
print(f'.npy: {len(vid_npy)} video | .csv: {len(vid_csv)} video')
print('Có map, THIẾU features:', sorted(vid_csv - vid_npy))
print('Có features, THIẾU map:', sorted(vid_npy - vid_csv))

# --- 2. Dựng lại ID theo đúng cột n (1-based, lấy thẳng từ CSV) ---
print('\nĐang đọc... (~1-2 phút vì file nằm trên Drive)')
all_vecs, all_ids, lech = [], [], []
for vid in sorted(vid_npy & vid_csv):
    vecs = np.load(f'{CLIP_DIR}/{vid}.npy').astype('float32')
    ns = pd.read_csv(f'{MAP_DIR}/{vid}.csv', usecols=['n'])['n'].tolist()
    if len(ns) != len(vecs):
        lech.append((vid, len(vecs), len(ns)))
        continue
    all_vecs.append(vecs)
    all_ids += [f'{vid}#{n:06d}' for n in ns]

X = np.concatenate(all_vecs)
print(f'\nX = {X.shape} | {len(all_ids)} id | video lệch số dòng: {len(lech)}')
if lech:
    print('Lệch (video, số vector, số dòng csv):', lech[:10])
print('ID đầu :', all_ids[0])
print('ID cuối:', all_ids[-1])

NameError: name 'MAP_DIR' is not defined

In [ ]:
# === BƯỚC 2: chuẩn hóa L2 + build FAISS IndexFlatIP ===
import faiss
faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
print('Index size:', index.ntotal)

faiss.write_index(index, f'{OUT_DIR}/faiss_clip.index')
np.save(f'{OUT_DIR}/keyframe_ids.npy', np.array(all_ids))
print('Đã lưu về', OUT_DIR)

In [ ]:
# === BƯỚC 3: smoke test — encode 1 câu text và tìm thử ===
# QUAN TRỌNG: model text PHẢI khớp model BTC dùng tạo CLIPFeatures.
# Thử lần lượt các ứng viên, model nào cho kết quả hợp lý là đúng.
%pip -q install open_clip_torch torch
import open_clip, torch

MODEL_NAME, PRETRAINED = 'ViT-B-32', 'openai'   # TODO: đổi nếu chiều vector không khớp
model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

q = 'a person eating ice cream near the sea'
with torch.no_grad():
    tv = model.encode_text(tokenizer([q])).float().numpy()
faiss.normalize_L2(tv)
D, I = index.search(tv, 10)
for d, i in zip(D[0], I[0]):
    print(f'{d:.3f}  {all_ids[i]}')
# Mở vài keyframe tương ứng xem có đúng ngữ nghĩa không → nếu lung tung, đổi MODEL_NAME.

In [ ]:
import os
data = '/content/drive/MyDrive/AIC2026/data'

# 1. Làm phẳng các thư mục lồng trùng tên (clip_features/clip-features -> clip_features)
for outer in ['clip_features', 'keyframes', 'metadata', 'objects']:
    op = f'{data}/{outer}'
    if not os.path.isdir(op):
        continue
    items = os.listdir(op)
    if len(items) == 1 and os.path.isdir(f'{op}/{items[0]}'):
        inner = f'{op}/{items[0]}'
        tmp = f'{data}/__tmp_{outer}'
        os.rename(inner, tmp)   # nhấc ruột ra ngoài
        os.rmdir(op)            # bỏ vỏ rỗng
        os.rename(tmp, op)      # đặt lại đúng tên chuẩn
        print('✓ đã làm phẳng:', outer)

# 2. Chuẩn hóa tên map-keyframes (gạch ngang theo quy ước của đội)
if os.path.isdir(f'{data}/map_keyframes') and not os.path.isdir(f'{data}/map-keyframes'):
    os.rename(f'{data}/map_keyframes', f'{data}/map-keyframes')
    print('✓ đổi tên map_keyframes -> map-keyframes')

# 3. Kiểm tra lại
print()
for sub in sorted(os.listdir(data)):
    p = f'{data}/{sub}'
    if os.path.isdir(p):
        files = os.listdir(p)
        print(f'{sub:20s} {len(files):7,} mục — ví dụ: {files[:3]}')

In [ ]:
import faiss, torch, numpy as np, open_clip
from PIL import Image
from IPython.display import display

KF_DIR = '/content/drive/MyDrive/AIC2026/data/keyframes'
q = 'a news anchor in a television studio'   # đổi câu tùy thích

def show_top(model_name, k=5):
    model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained='openai')
    tok = open_clip.get_tokenizer(model_name)
    with torch.no_grad():
        tv = model.encode_text(tok([q])).float().numpy()
    faiss.normalize_L2(tv)
    D, I = index.search(tv, k)
    print(f'\n=== {model_name} — "{q}" ===')
    for d, i in zip(D[0], I[0]):
        kf = all_ids[i]; vid, n = kf.split('#')
        print(f'{d:.3f}  {kf}')
        try:
            display(Image.open(f'{KF_DIR}/{vid}/{int(n):03d}.jpg').resize((320, 180)))
        except FileNotFoundError:
            print('   (chưa thấy ảnh — kiểm tra tên file keyframe)')

show_top('ViT-B-32-quickgelu')
show_top('ViT-B-16')

In [ ]:
import os, glob, numpy as np, pandas as pd

# Xem quanh CLIP_DIR để tìm thư mục map-keyframes
p1 = os.path.dirname(CLIP_DIR.rstrip('/'))
p2 = os.path.dirname(p1)
cand = []
for p in [p1, p2]:
    try:
        items = sorted(os.listdir(p))
    except Exception:
        continue
    print(f'{p}:\n   {items[:30]}')
    cand += [os.path.join(p, d) for d in items
             if 'map' in d.lower() and os.path.isdir(os.path.join(p, d))]

# Gỡ trường hợp lồng thư mục (giống lỗi clip_features/clip-features trước đây)
MAP_DIR = None
for c in cand:
    if glob.glob(f'{c}/*.csv'):
        MAP_DIR = c; break
    for sub in glob.glob(f'{c}/*'):
        if os.path.isdir(sub) and glob.glob(f'{sub}/*.csv'):
            MAP_DIR = sub; break
    if MAP_DIR: break

if not MAP_DIR:
    print('\n>> Chưa thấy map-keyframes. Gửi Claude phần danh sách in ở trên.')
else:
    files = sorted(glob.glob(f'{MAP_DIR}/*.csv'))
    print(f'\nMAP_DIR = {MAP_DIR}   ({len(files)} file csv)')
    df = pd.read_csv(files[0])
    vid = os.path.splitext(os.path.basename(files[0]))[0]
    print('cột      :', list(df.columns))
    print('n đầu    :', df['n'].iloc[0], '| n cuối:', df['n'].iloc[-1], '| số dòng csv:', len(df))
    npy = os.path.join(CLIP_DIR, vid + '.npy')
    if os.path.exists(npy):
        print('số dòng .npy:', np.load(npy, mmap_mode='r').shape[0], '  <-- phải bằng số dòng csv')